# Import

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

## Test OpenAI API

In [3]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely — you can usually join a course after it has started, but it depends on the course rules and how far along it is.

A few quick things to check:
- **Enrollment deadline:** some courses allow late joining, others don’t.
- **Missed content:** you may need to catch up on earlier lessons.
- **Instructor approval:** some courses require permission to join late.
- **Platform access:** make sure you can still access recordings, assignments, and materials.

If you want, I can help you draft a short message to the course organizer asking if late enrollment is possible.


In [4]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [5]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [6]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


# Fetch courses

In [7]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

Fetch course FAQ details

In [8]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

Check on the first item

In [9]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

# Minsearch

In [10]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

Trying a search

In [11]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

Show all matching questions

In [12]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

Boosting fields & filter by course

In [13]:
results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [14]:
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

# Wrapper search function

In [15]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}
    
    results = index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
    return results

In [16]:
[search_results for search_results in search(question)]

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

System Prompt

In [17]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

User Prompt template

In [18]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

Building the context

In [19]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        question = doc["question"]
        section = doc["section"]
        answer = doc["answer"]

        line = f"Section: {section}\nQ: {question}\nA: {answer}"
        lines.append(line)

    return "\n".join(lines)

Building the prompt

In [20]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

## Test prompts

In [21]:
prompt = build_prompt(question, search(question))
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
Section: General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
Section: General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
Section: General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) afte

# Sending the prompt to the LLM

In [22]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

## Explore the response

In [23]:
response.output[0]

ResponseOutputMessage(id='msg_04c11c4a7e618011006a3986f77420819e82bd313d05ad5115', content=[ResponseOutputText(annotations=[], text='Yes — you can join now.\n\nYou can start learning and submitting homework as long as the course submission form is still open. If you want a certificate, make sure you submit your project while submissions are being accepted.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [24]:
response.output[0].content[0].text

'Yes — you can join now.\n\nYou can start learning and submitting homework as long as the course submission form is still open. If you want a certificate, make sure you submit your project while submissions are being accepted.'

In [25]:
response.output_text

'Yes — you can join now.\n\nYou can start learning and submitting homework as long as the course submission form is still open. If you want a certificate, make sure you submit your project while submissions are being accepted.'

In [26]:
response.usage

ResponseUsage(input_tokens=490, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=47, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=537)

# Calculate Token Costs

In [27]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000579

# The LLM function

In [28]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

# The Full RAG

In [29]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

## Test

In [30]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still open.


In [31]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a live cohort and pass the capstone project.\n\nA few important points:\n- Self-paced mode does not include a certificate.\n- Homework is not mandatory for the certificate.\n- You must be able to peer-review 3 capstones during the live course period.\n\nAlso, make sure your official name is set in your course profile, since that name will appear on the certificate.'

In [36]:
rag("Will there be a session on guardrails?")

NameError: name 'openai_client' is not defined

# Debug Jupyter Notebook not responding after idle.

In [35]:
for name, value in list(globals().items()):
    if not name.startswith("_"):
        try:
            print(name, type(value), end=" ")
            repr(value)
            print("ok")
        except Exception as e:
            print("bad:", e)

In <class 'list'> ok
Out <class 'dict'> ok
get_ipython <class 'method'> ok
exit <class 'IPython.core.autocall.ZMQExitAutocall'> ok
quit <class 'IPython.core.autocall.ZMQExitAutocall'> ok
open <class 'function'> ok
load_dotenv <class 'function'> ok
OpenAI <class 'type'> ok
llm <class 'function'> ok
question <class 'str'> ok
answer <class 'str'> ok
context <class 'str'> ok
prompt <class 'str'> ok
requests <class 'module'> ok
docs_url <class 'str'> ok
courses_raw <class 'list'> ok
documents <class 'list'> ok
url_prefix <class 'str'> ok
course <class 'dict'> ok
course_url <class 'str'> ok
course_data <class 'list'> ok
Index <class 'type'> ok
index <class 'minsearch.minsearch.Index'> ok
search_results <class 'list'> ok
results <class 'list'> ok
search <class 'function'> ok
INSTRUCTIONS <class 'str'> ok
USER_PROMPT_TEMPLATE <class 'str'> ok
build_context <class 'function'> ok
build_prompt <class 'function'> ok
input_price <class 'float'> ok
output_price <class 'float'> ok
cost <class 'float'

In [34]:
del openai_client
del response
del course_response

# T